# 11c — Manifest v4: the re-registered design (study plan v0.20; methods_log M2.12 / M4.38; zero solves)

Supersedes manifest v3.1 (kept as the disclosed prior registration, never deleted). Trigger: the post-freeze sequence (E20) and
the corridors-out isolation (PF-4) showed that the two-layer connectivity block halved each layer's expressivity while both are
live, spatially independent values; that structural connectivity's pinch points pin at balanced weights only under a convex shape
with no target; and that 1/v's tail top is partly a v → 0 artifact.

**The v4 design.** Five discretionary blocks at 20% each — core habitat (macrorefugia, 1/v WITH a velocity floor per
realization), structural connectivity (transboundary raw current, CONVEX I², t = 1, no target: the R2 semantic gate), climate
corridors (Carroll centrality, identity), carbon (m_soc θ-target + biomass, 74.2 / 25.8), biodiversity (birds + mammals, equal).
The representativeness block (20 features, window-derived targets) and naturalness (w = 1, outside) are unchanged. Seven
scenarios (balanced; core-habitat-, structural-connectivity-, climate-corridors-, biodiversity-, carbon-forward; naturalness
push) × two refugia futures = **14 voting cells**. Constants unchanged: k = 50, g = 5%, opt_gap 1e-4, NumericFocus 2, per-block
floors at 95% of the anchor, estimator `mga_maxham_v1`, verdict rule v2.

**This notebook = the spec's pre-solve steps (1)–(4):** (1) refugia velocity pre-checks (existing minimum, near-zero count),
the floor applied per realization and 1/v re-derived; (2) the re-audit of floored refugia and squared transboundary current
under the frozen constants (feature cards; the semantic gate withholds transboundary's target); (3) five-block accounting at
20% and the seven scenarios' (w, t) pairs; (4) manifest v4 frozen with its sha. It also builds the v4 stack
(`input_data/aligned_stack_v4/`: unchanged layers byte-identical to `aligned_stack/`, the two re-shaped layers written fresh,
its own `manifest.json`), which every flagship reader uses through `config.Y2Y_STACK_DIR` under `config.Y2Y_VERSION = "v4"`.

Run order after this: **12** (anchors, twins, the reference cell's unguarded MGA) → 13 → 15 → **18** (guarded sweep, all 14) →
**18b** → 18c → 19 → 20 → 21. Ethan runs every notebook in VS Code. Minutes; zero solves. Write-once: an existing frozen
manifest v4 whose sha matches its freeze file is never rewritten.


In [ ]:
# ---- bootstrap ----------------------------------------------------------------------------------------------
import importlib, json, hashlib, pathlib, shutil, sys
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import rasterio
_cands = [p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "config.py").exists()]
assert _cands, "config.py not found above the notebook"
ROOT = _cands[0]; sys.path.insert(0, str(ROOT))
import config, leverage_core as lc
for _m in (config, lc):
    importlib.reload(_m)
assert config.Y2Y_VERSION == "v4", f"config.Y2Y_VERSION is {config.Y2Y_VERSION}: manifest v4 is built with the v4 switch active"
V31, V4 = config.y2y_paths("v3.1"), config.y2y_paths("v4")
SRC, DST = V31.stack, V4.stack                                   # aligned_stack/ (prior registration)  ->  aligned_stack_v4/
assert DST == config.Y2Y_STACK_DIR and V4.efg_subdir == "iucn_efg_v3" and (SRC / V4.efg_subdir).is_dir(), (DST, V4.efg_subdir)
SPEC = ROOT / "analyses" / "y2y" / "spec"; REC = V4.records; REC.mkdir(exist_ok=True, parents=True)
CARDS = ROOT / "analyses" / "y2y" / "audit" / "feature_cards_v4"; CARDS.mkdir(exist_ok=True, parents=True)
BLK = config.BLOCKS_FIVE; assert config.BLOCKS is BLK
CONT = lc.continuous_features()                                  # the 8 continuous features in manifest order
sha = lambda p: hashlib.sha256(pathlib.Path(p).read_bytes()).hexdigest()
def read(p):
    with rasterio.open(p) as s:
        return s.read(1, masked=True).astype("float64").filled(np.nan), s.profile
def write_like(profile, arr, out):
    prof = dict(profile); prof.update(dtype="float32", nodata=np.nan, count=1)
    out.parent.mkdir(parents=True, exist_ok=True)
    with rasterio.open(out, "w", **prof) as d:
        d.write(np.where(np.isfinite(arr), arr, np.nan).astype(np.float32), 1)
PU = lc.pu_mask(SRC)
with rasterio.open(SRC / "mask_protected_areas.tif") as s:
    LOCKED = (s.read(1) == 1) & PU
DISC = PU & ~LOCKED
print(f"VERSION {V4.version} | stack {SRC.name} -> {DST.name} | PU {int(PU.sum()):,} cells, allocatable {int(DISC.sum()):,} | blocks {list(BLK)}")


In [ ]:
# ---- (1) refugia velocity pre-checks + the floor per realization -----------------------------------------------------------
# Backward climate velocity (AdaptWest, km/yr) = distance from a cell's future climate to its nearest current analog, divided by
# the elapsed span: 1961-1990 -> 2071-2100 is 110 yr midpoint to midpoint (the spec's "~80 yr" was a placeholder). The floor is
# the slowest velocity the 1 km grid can resolve over that span: one cell width / 110 yr. Cells below it are set TO the floor
# ("analog within one cell" refugia all carry the same maximal 1/v) -- numerics hygiene, the mirror of dust thresholding.
SPAN_YR = 110.0; V_FLOOR = 1.0 / SPAN_YR                        # km/yr
RAWP = {"585": config.ALIGNED_DIR / "climate_scenarios" / "585_2071_2100.tif", "245": config.ALIGNED_DIR / "climate_scenarios" / "245_2071_2100.tif"}
STKP = {"585": SRC / "climate_type_macrorefugia.tif", "245": SRC / "climate_realizations" / "macrorefugia_245_2071_2100.tif"}
OUTP = {"585": DST / "climate_type_macrorefugia.tif", "245": DST / "climate_realizations" / "macrorefugia_245_2071_2100.tif"}
rows, FLOORED = [], {}
for lv in ("585", "245"):
    v, _ = read(RAWP[lv]); x, prof = read(STKP[lv]); vp, xp = v[PU], x[PU]
    assert np.isfinite(vp).all() and (vp > 0).all(), f"{lv}: raw velocity has non-finite or non-positive cells on the PU"
    assert np.nanmax(np.abs(1.0 / vp - xp)) < 1e-6, f"{lv}: the registered stack layer is not 1/v of the raw realization"
    n_at = {k: int((vp <= m * V_FLOOR).sum()) for k, m in (("le_floor", 1), ("le_2x_floor", 2), ("le_5x_floor", 5), ("le_10x_floor", 10))}
    floored = np.where(np.isfinite(x), np.minimum(x, 1.0 / V_FLOOR), np.nan)               # cap 1/v at 1/floor == floor v; mask + dust untouched
    assert np.allclose(floored[PU], 1.0 / np.maximum(vp, V_FLOOR), rtol=1e-6), f"{lv}: floored layer != 1/max(v, floor)"
    changed = int((floored[PU] != xp).sum())
    rows.append(dict(realization=f"ssp{lv}_2071_2100", span_yr=SPAN_YR, v_floor_km_per_yr=round(V_FLOOR, 6), min_v_km_per_yr=float(vp.min()),
                     p01_v=float(np.percentile(vp, 1)), median_v=float(np.median(vp)), n_zero=int((vp == 0).sum()), **n_at,
                     max_inv_v_registered=float(xp.max()), max_inv_v_floored=float(np.nanmax(floored[PU])), cells_changed_by_floor=changed,
                     floor_inert=bool(changed == 0)))
    FLOORED[lv] = (floored, prof)
PRE = pd.DataFrame(rows); PRE.to_csv(REC / "velocity_precheck.csv", index=False)
pd.set_option("display.width", 220); print(PRE.to_string(index=False))
for lv in ("585", "245"):
    write_like(FLOORED[lv][1], FLOORED[lv][0], OUTP[lv])
print("\nfloor verdict:", "INERT on both realizations -- the product already carries a minimum velocity above one cell width per span; the v4 layers equal the registered 1/v cell for cell"
      if PRE.floor_inert.all() else "ACTIVE -- cells at/below the floor were capped; see cells_changed_by_floor")
print(f"wrote {OUTP['585'].relative_to(ROOT)} and {OUTP['245'].relative_to(ROOT)}")


In [ ]:
# ---- the squared transboundary current (I^2) + the registered dust rule; then the v4 stack copy + its manifest ------------------
# Structural connectivity is valued CONVEXLY (spec v0.20: raw cumulative Circuitscape current, max-to-median 46, a real pinch-point
# tail where flow has few alternative paths -- I^2 encodes severance-over-proportional-loss). Squaring pushes float residue toward
# the matrix range the 2026-08-26 dust rule exists for, so the rule (cells below DUST_SHARE_MIN of the feature total -> 0) is
# re-applied to the squared layer and its effect disclosed. Scale is irrelevant: 03 sum-normalizes every feature.
conn, prof = read(SRC / "transboundary_connectivity.tif")
sq = np.where(np.isfinite(conn), conn.astype(np.float64) ** 2, np.nan)
tot = float(np.nansum(sq)); dust = np.isfinite(sq) & (sq > 0) & (sq < config.DUST_SHARE_MIN * tot)
n_dust, m_dust = int(dust.sum()), float(np.nansum(sq[dust])); sq[dust] = 0.0
assert (m_dust / tot) < 1e-4, "dust rule dropped real mass on the squared layer -- STOP"
write_like(prof, sq, DST / "transboundary_connectivity.tif")
r_reg, r_sq = float(np.nanmax(conn[PU]) / np.nanmean(conn[PU])), float(np.nanmax(sq[PU]) / np.nanmean(sq[PU]))
print(f"transboundary I^2: dust zeroed {n_dust:,} cells ({m_dust / tot:.1e} of the squared mass) | max/mean on the PU {r_reg:.1f} (registered) -> {r_sq:.1f} (squared)")
# the rest of the stack: byte-identical copies (asserted by sha256); the E20/E9 test layers and the v1 block folder stay behind
CHANGED = {"climate_type_macrorefugia.tif", "transboundary_connectivity.tif"}
HASHES = {"changed": {}, "unchanged": {}, "realizations": {}, "efg": {}}
for p in sorted(SRC.glob("*.tif")):
    if p.name in CHANGED or p.name.startswith(("e20_", "e9_")):
        continue
    q = DST / p.name
    if not q.exists() or sha(q) != sha(p):
        shutil.copy2(p, q)
    assert sha(q) == sha(p), p.name
    HASHES["unchanged"][p.stem] = sha(q)
for name in CHANGED:
    HASHES["changed"][pathlib.Path(name).stem] = dict(v4=sha(DST / name), v31=sha(SRC / name))
for p in sorted((SRC / "climate_realizations").glob("*.tif")):
    q = DST / "climate_realizations" / p.name
    if p.name == "macrorefugia_245_2071_2100.tif":
        HASHES["realizations"][p.stem] = dict(v4=sha(q), v31=sha(p))              # written by cell 2 (floored)
        continue
    if not q.exists() or sha(q) != sha(p):
        q.parent.mkdir(exist_ok=True); shutil.copy2(p, q)
    HASHES["realizations"][p.stem] = sha(q)
efg_dst = DST / V4.efg_subdir; efg_dst.mkdir(exist_ok=True)
for p in lc.efg_paths(SRC):
    q = efg_dst / p.name
    if not q.exists() or sha(q) != sha(p):
        shutil.copy2(p, q)
    assert sha(q) == sha(p), p.name
    HASHES["efg"][p.stem] = sha(q)
for p in sorted(SRC.glob("*.gpkg")):                                                   # ROI / lock-in vectors of the sub-analyses (unused by y2y)
    q = DST / p.name
    if not q.exists() or sha(q) != sha(p):
        shutil.copy2(p, q)
assert len(HASHES["efg"]) == 20 and len(HASHES["unchanged"]) >= 9, (len(HASHES["efg"]), len(HASHES["unchanged"]))
(REC / "stack_v4_layer_sha256.json").write_text(json.dumps(dict(source=str(SRC.relative_to(ROOT)), stack=str(DST.relative_to(ROOT)),
    dust_rule=dict(share_min=config.DUST_SHARE_MIN, transboundary_sq_cells_zeroed=n_dust, transboundary_sq_mass_share=m_dust / tot),
    velocity_floor=dict(span_yr=SPAN_YR, v_floor_km_per_yr=V_FLOOR), created_utc=datetime.now(timezone.utc).isoformat(), **HASHES), indent=1))
mpath = config.write_manifest(analysis="y2y")                                          # -> aligned_stack_v4/manifest.json (Y2Y_STACK_DIR under v4)
MJ = json.loads(pathlib.Path(mpath).read_text()); roles = pd.Series([l["role"] for l in MJ["layers"]]).value_counts().to_dict()
assert all(pathlib.Path(ROOT / l["path"]).exists() and str(l["path"]).startswith(str(DST.relative_to(ROOT))) for l in MJ["layers"]), "manifest points outside the v4 stack"
print(f"v4 stack: {len(HASHES['unchanged'])} unchanged layers (sha-identical), 2 re-shaped, {len(HASHES['efg'])} EFG features, {len(HASHES['realizations'])} realization layer(s) | manifest {pathlib.Path(mpath).relative_to(ROOT)}: {roles}")


In [ ]:
# ---- (2) the re-audit under the frozen constants: floored refugia (both realizations) + squared transboundary current -----------
# R1-R4 as at Gate 0a (config.AUDIT: theta 5x, a_min 0.5%, t_min 0.15, lambda 0.10), on the v4 stack. Expected (spec): refugia
# diffuse-linear, unchanged; transboundary crosses theta / a_min (concentrated-satiating, R2 target ~0.198) -- and the R2 SEMANTIC
# GATE, now in the registered protocol, withholds the target: pinch points are PLACES (the claim is the set of cells), secured by
# the convex shape on the linear arm with t = 1, where the pull persists inside the band (PF-2 showed the target un-pins them).
PU4 = lc.pu_mask(DST); assert (PU4 == PU).all(), "the v4 stack changed the planning-unit mask"
GATE = {"climate_type_macrorefugia": ("weight (diffuse-linear)", "value = residence time within the horizon, linear; floor = numerics hygiene"),
        "transboundary_connectivity": ("weight, t = 1 -- R2 SEMANTIC GATE withholds the target", "pinch points are place-semantic: the claim IS the set of cells; a target secures an amount and un-pins them (PF-2: 100% -> 7%)")}
rows = []
for name in ("climate_type_macrorefugia", "transboundary_connectivity"):
    r = lc.classify(name, handoff_dir=DST); r31 = lc.classify(name, handoff_dir=SRC)
    r.update(realization="ssp585" if name.startswith("climate") else "-", shape={"climate_type_macrorefugia": "1/max(v, floor)", "transboundary_connectivity": "I^2"}[name],
             registered_lever=GATE[name][0], gate_reason=GATE[name][1], v31_cls=r31["cls"], v31_leverage=r31["leverage"], v31_theta_area=r31["theta_area"], v31_target=r31["target"])
    rows.append(r); lc.feature_card(name, CARDS, handoff_dir=DST)
x245 = read(OUTP["245"])[0][PU]; cls, lever, target, d = lc.classify_values(x245)
rows.append(dict(feature="climate_type_macrorefugia", orient="reciprocal", realization="ssp245", shape="1/max(v, floor)", cls=cls, lever=lever, target=target, **d,
                 registered_lever=GATE["climate_type_macrorefugia"][0], gate_reason=GATE["climate_type_macrorefugia"][1]))
AUD = pd.DataFrame(rows); AUD.to_csv(REC / "audit_cards_v4.csv", index=False)
print(AUD[["feature", "realization", "shape", "cls", "lever", "target", "leverage", "theta_area", "theta_target", "v31_cls", "v31_leverage", "registered_lever"]].to_string(index=False))
ref = AUD[(AUD.feature == "climate_type_macrorefugia")]; tb = AUD[AUD.feature == "transboundary_connectivity"].iloc[0]
print(f"\nrefugia: {' / '.join(ref.cls)} (expected diffuse-linear, unchanged) | transboundary I^2: {tb.cls}, R2 target {tb.target} -> registered lever: {tb.registered_lever}")
print(f"cards -> {CARDS.relative_to(ROOT)}/ ; table -> {REC.relative_to(ROOT)}/audit_cards_v4.csv")


In [ ]:
# ---- (3) five-block accounting at 20% each; the seven scenarios' (w, t) pairs under constant intended influence --------------
# Doubling rule unchanged: the forward block takes 40%, the other four 15% each. S4 carbon-forward also moves the m_soc target
# 0.332 -> 0.552 (theta 3x); S5 = S0 with naturalness pushed x10 (documented inexpressible, runs as in v3.1). Carbon's inner split
# is the measured mass-proportional 74.2 / 25.8 (scenarios_v2); biodiversity equal (the standing convention). Weights are derived
# per climate cell from the v4 stack (the 245 cells swap the floored 245 refugia layer in), realized == intended asserted.
SC2 = json.loads((SPEC / "scenarios_v2.json").read_text()); WITHIN = {"carbon": SC2["S0_balanced"]["within_block"]["carbon"]}
T0, T4 = {"irrecoverable_carbon_m_soc": 0.332}, {"irrecoverable_carbon_m_soc": 0.552}
assert T0 == SC2["S0_balanced"]["targets"] and T4 == SC2["S4_carbon"]["targets"]
def shares(forward=None):
    return {b: (0.4 if b == forward else (0.15 if forward else 0.2)) for b in BLK}
SCEN = {"s0": dict(name="S0_balanced", label="Balanced", shares=shares(), targets=T0, extra={}, regime="theta5_amount"),
        "s1": dict(name="S1_core_habitat", label="Core-habitat-forward", shares=shares("core_habitat"), targets=T0, extra={}, regime="theta5_amount"),
        "s2": dict(name="S2_structural_connectivity", label="Structural-connectivity-forward", shares=shares("structural_connectivity"), targets=T0, extra={}, regime="theta5_amount"),
        "s2c": dict(name="S2c_climate_corridors", label="Climate-corridors-forward", shares=shares("climate_corridors"), targets=T0, extra={}, regime="theta5_amount"),
        "s3": dict(name="S3_biodiversity", label="Biodiversity-forward", shares=shares("biodiversity"), targets=T0, extra={}, regime="theta5_amount"),
        "s4": dict(name="S4_carbon", label="Carbon-forward", shares=shares("carbon"), targets=T4, extra={}, regime="theta3_places"),
        "s5": dict(name="S5_naturalness", label="Naturalness push (S0 + gHM x10)", shares=shares(), targets=T0, extra={"human_modification": 10.0}, regime="theta5_amount")}
LP = {"ssp585_2071_2100": None, "ssp245_2071_2100": {"climate_type_macrorefugia": OUTP["245"]}}
W = {}
for sid, s in SCEN.items():
    for climate, lp in LP.items():
        d = lc.scenario_weights(s["shares"], within_block=WITHIN, targets=s["targets"], blocks=BLK, layer_paths=lp, handoff_dir=DST)
        w = {r.feature: round(float(r.w), 6) for r in d.itertuples()}; w.update(s["extra"])
        W[(sid, climate)] = dict(weights=w, intended={r.feature: round(float(r.intended_share), 6) for r in d.itertuples()},
                                 swing={r.feature: float(r.per_unit_swing) for r in d.itertuples()})
ORDER = [f for f in CONT if f != "human_modification"] + ["human_modification"]
tab = pd.DataFrame({f"{sid}@{c.split('_')[0][-3:]}": W[(sid, c)]["weights"] for sid in SCEN for c in LP}).reindex(ORDER).fillna(1.0)   # naturalness outside at 1
print("weights per formulation (v4 stack; constant intended influence; naturalness outside at 1, x10 under S5):"); print(tab.to_string(float_format=lambda v: f"{v:.4f}"))
payload = dict(_meta=dict(derived_utc=datetime.now(timezone.utc).isoformat(), spec_version="v0.20 (manifest v4)", blocks=BLK, doubling_rule="forward block 0.40, the other four 0.15; balanced 0.20 x 5",
                          within_block=WITHIN, shapes={"climate_type_macrorefugia": f"1/max(v, {V_FLOOR:.6f} km/yr)", "transboundary_connectivity": "I^2 (t = 1, R2 semantic gate)", "others": "identity"},
                          stack=str(DST.relative_to(ROOT)), normalization="derived weights rescaled to mean 1 over the blocked features; outside features at their baselines"),
               **{s["name"]: dict(scenario_id=sid, label=s["label"], block_shares=s["shares"], within_block=WITHIN, targets=s["targets"], extra=s["extra"], carbon_regime=s["regime"],
                                   weights={c: W[(sid, c)]["weights"] for c in LP}, intended_shares=W[(sid, "ssp585_2071_2100")]["intended"]) for sid, s in SCEN.items()})
(SPEC / "scenarios_v4.json").write_text(json.dumps(payload, indent=2))
print(f"\nwrote {(SPEC / 'scenarios_v4.json').relative_to(ROOT)} ({len(SCEN)} scenarios x {len(LP)} climate cells)")


In [ ]:
# ---- (4) manifest v4: 14 design formulations, frozen with its sha (write-once) ------------------------------------------------
M31 = pd.read_csv(V31.manifest).set_index("formulation_id"); sha_v31 = sha(V31.manifest)
assert sha_v31 == V31.freeze.read_text().split()[0], "manifest v3.1 no longer matches its freeze hash -- stop"
T_EFG = {k: v for k, v in json.loads(M31.iloc[0].target_vector).items() if k not in CONT}          # the 20 window-derived EFG targets, unchanged
assert len(T_EFG) == 20 and all({k: v for k, v in json.loads(r.target_vector).items() if k not in CONT} == T_EFG for _, r in M31.iterrows())
EFG_SHA = hashlib.sha256("\n".join(f"{k}:{v}" for k, v in sorted(HASHES["efg"].items())).encode()).hexdigest()
assert EFG_SHA == M31.iloc[0].efg_block_sha256, "the v4 stack's EFG block differs from v3.1's"
SHAPES = payload["_meta"]["shapes"]; REF = "s0_ssp585_theta5"; PROBES = [0.02, 0.05, 0.10]
v31 = M31.loc[REF]; now = datetime.now(timezone.utc).isoformat(); rows = []
for i, (climate, lp) in enumerate(LP.items()):
    for j, (sid, s) in enumerate(SCEN.items()):
        fid = f"{sid}_{climate.split('_')[0]}_{s['regime'].split('_')[0]}"
        hashes = {**{k: v for k, v in HASHES["unchanged"].items() if k in CONT}, "transboundary_connectivity": HASHES["changed"]["transboundary_connectivity"]["v4"],
                  "climate_type_macrorefugia": HASHES["changed"]["climate_type_macrorefugia"]["v4"] if lp is None else HASHES["realizations"]["macrorefugia_245_2071_2100"]["v4"], **HASHES["efg"]}
        rows.append(dict(formulation_id=fid, scenario_id=sid, scenario_name=s["name"], climate_level=climate, carbon_regime=s["regime"], budget_pct=config.BUDGET_PCT,
                         weight_vector=json.dumps(W[(sid, climate)]["weights"]), target_vector=json.dumps({**s["targets"], **T_EFG}),
                         influence_profile_intended=json.dumps(W[(sid, climate)]["intended"]), k_requested=50, band_gap_g=0.05, floor_g=0.05, opt_gap=1e-4, numeric_focus=2,
                         dust_rule_version=v31.dust_rule_version, role="design", estimator=v31.estimator, verdict_rule=v31.verdict_rule, solver="gurobi", seed_policy=v31.seed_policy,
                         input_layer_hashes=json.dumps(hashes), kbest_ref="", twin_ref="",
                         block_structure=json.dumps(BLK), value_shapes=json.dumps(SHAPES),
                         macrorefugia_path=str((OUTP["585"] if lp is None else OUTP["245"]).relative_to(ROOT)),
                         reference_cell=bool(fid == REF), unguarded_probes=json.dumps(PROBES if fid == REF else []), e12_seed=20260910 + i * len(SCEN) + j,
                         manifest_version=4, efg_block_version="v3", efg_block_sha256=EFG_SHA, efg_features=json.dumps(sorted(HASHES["efg"])), efg_target_rule=v31.efg_target_rule,
                         supersedes=f"v3.1 {sha_v31[:16]}", trigger="study plan v0.20 (E20 / PF-4, R10.24-R10.26): five blocks at 20%, transboundary I^2 (no target), refugia velocity floor",
                         created_utc=now, frozen=True))
M4 = pd.DataFrame(rows); assert M4.formulation_id.is_unique and len(M4) == 14 and M4.reference_cell.sum() == 1
for fn in ("efg_targets.json", "efg_window_footprints.csv"):                       # the EFG-target record the readers expect beside the version's records (unchanged from v3.1)
    if not (REC / fn).exists():
        shutil.copy2(V31.records / fn, REC / fn)
if V4.manifest.exists() and V4.freeze.exists() and sha(V4.manifest) == V4.freeze.read_text().split()[0]:
    d4 = V4.freeze.read_text().split()[0]
    print(f"manifest v4 already frozen ({d4[:16]}...) -- kept byte-identical (supersede, never delete); delete both files to re-freeze")
else:
    M4.to_csv(V4.manifest, index=False); d4 = sha(V4.manifest); V4.freeze.write_text(f"{d4}  {V4.manifest.name}\n")
    print(f"FROZEN: {V4.manifest.relative_to(ROOT)} (14 design formulations; blocks {list(BLK)}; EFG block sha {EFG_SHA[:16]}...) | sha256 {d4[:16]}...")
print(M4[["formulation_id", "scenario_id", "climate_level", "carbon_regime", "reference_cell", "unguarded_probes", "macrorefugia_path"]].to_string(index=False))
print("\nnext: 12 (VERSION v4: anchors + twins for all 14, unguarded MGA on the reference cell at g = 2/5/10%) -> 13 -> 15 -> 18 -> 18b -> 18c -> 19 -> 20 -> 21")
